In [ ]:
# ── SEL 1: INSTALASI ──────────────────────────────────────
# Estimasi waktu: 2-3 menit. Tunggu sampai "SELESAI".

# 1. Install dependency sistem & Ollama
!apt-get install -y zstd curl lsof -q 2>/dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Install Open WebUI
!pip install open-webui -q

# 3. Install Cloudflare Tunnel (cloudflared)
import os, shutil
CF_BIN = "/usr/local/bin/cloudflared"
if not shutil.which("cloudflared"):
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o {CF_BIN} && chmod +x {CF_BIN}
else:
    print("  ✅ cloudflared sudah terinstall")

# 4. Verifikasi semua dependency
missing = []
for cmd in ["ollama", "cloudflared", "open-webui"]:
    if not shutil.which(cmd):
        missing.append(cmd)
if missing:
    print(f"❌ GAGAL: {', '.join(missing)} tidak ditemukan setelah instalasi")
else:
    print("────────────────────────────────")
    print("✅ SELESAI — Semua dependency terinstall. Lanjut ke Sel 2.")
    print("────────────────────────────────")

In [ ]:
# ── SEL 2: SETUP & JALANKAN OLLAMA ────────────────────────
# Ganti model di bawah ini jika perlu:
MODEL = "qwen2.5-coder:7b"   # default, 4.5 GB
# MODEL = "qwen2.5-coder:14b"  # vibe coding, 8.9 GB
# MODEL = "deepseek-coder:14b" # vibe coding, 8.5 GB
# MODEL = "deepseek-r1:7b"     # reasoning, 4.5 GB
# MODEL = "gemma4:12b"         # vibe coding
# MODEL = "llama3.1:8b"        # chat umum, 5 GB

import subprocess, time, os, shutil, json, urllib.request, sys
from google.colab import drive

# ─── Mount Google Drive (hanya jika belum) ───
print("[1/4] Mounting Google Drive...")
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')
else:
    print("  ✅ Drive sudah ter-mount sebelumnya")

# ─── Set up model paths ───
print("[2/4] Menyiapkan folder model...")
DRIVE_PATH = "/content/drive/MyDrive/ollama_models"
LOCAL_PATH = os.path.expanduser("~/.ollama/models")

# Restore model dari Drive ke lokal (jika ada)
RESTORED_FLAG = os.path.join(LOCAL_PATH, ".restored")
if os.path.exists(DRIVE_PATH) and os.listdir(DRIVE_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)
    if not os.path.exists(RESTORED_FLAG):
        print("  📦 Mengembalikan model dari Google Drive...")
        for item in os.listdir(DRIVE_PATH):
            src = os.path.join(DRIVE_PATH, item)
            dst = os.path.join(LOCAL_PATH, item)
            if not os.path.exists(dst):
                try:
                    if os.path.isdir(src):
                        shutil.copytree(src, dst, dirs_exist_ok=True)
                    else:
                        shutil.copy2(src, dst)
                except Exception as e:
                    print(f"    ⚠️  Gagal copy {item}: {e}")
        open(RESTORED_FLAG, 'w').close()
        print("  ✅ Model dipulihkan dari Drive")
    else:
        print("  ✅ Model sudah direstore sebelumnya (skip)")
elif not os.path.exists(LOCAL_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)

# Pakai local path (bukan Drive langsung) — lebih kompatibel dengan Ollama
os.environ["OLLAMA_MODELS"] = LOCAL_PATH

# ─── Kill sisa proses Ollama lama (jika ada) ───
print("[3/4] Membersihkan proses lama...")
os.system("pkill -f 'ollama serve' 2>/dev/null || true")
time.sleep(1)

# ─── Start Ollama ───
print("[4/4] Menjalankan Ollama & memuat model...")
ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"

if not os.path.exists(ollama_bin):
    print("❌ ERROR: Ollama tidak ditemukan. Jalankan Sel 1 dulu.")
else:
    # Bind ke 127.0.0.1 — lebih aman, tunnel tetap bisa akses
    os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
    proc_ollama = subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    # Tunggu Ollama siap
    print("  Menunggu Ollama siap...", end="", flush=True)
    ollama_ready = False
    for _ in range(30):
        try:
            r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            r.close()
            print(" ✅")
            ollama_ready = True
            break
        except Exception:
            print(".", end="", flush=True)
            time.sleep(2)
    if not ollama_ready:
        print(" ⚠️  Ollama tidak merespon — cek log di atas")

    # Pull model
    print(f"  Downloading model {MODEL} (mungkin butuh beberapa menit)...")
    pull = subprocess.Popen(
        [ollama_bin, "pull", MODEL],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in iter(pull.stdout.readline, ''):
        line = line.rstrip()
        if line:
            print(f"  {line}")
    pull.wait()

    if pull.returncode != 0:
        print(f"  ⚠️  Pull gagal (exit code {pull.returncode})")
    else:
        print("  ✅ Model berhasil di-pull")

    # Backup model ke Drive
    if os.path.exists(LOCAL_PATH):
        print("  💾 Backup model ke Google Drive...")
        os.makedirs(DRIVE_PATH, exist_ok=True)
        for item in os.listdir(LOCAL_PATH):
            if item.startswith('.'):
                continue
            src = os.path.join(LOCAL_PATH, item)
            dst = os.path.join(DRIVE_PATH, item)
            if not os.path.exists(dst):
                try:
                    if os.path.isdir(src):
                        shutil.copytree(src, dst, dirs_exist_ok=True)
                    else:
                        shutil.copy2(src, dst)
                except Exception as e:
                    print(f"    ⚠️  Gagal backup {item}: {e}")
        print("  ✅ Backup ke Drive selesai")

    print("────────────────────────────────")
    print(f"✅ MODEL {MODEL} SIAP — Lanjut ke Sel 3.")
    print("────────────────────────────────")

In [ ]:
# ── SEL 3: JALANKAN WEBUI & TUNNEL (CLOUDFLARE) ────────────
# SEL INI HARUS TERUS BERJALAN SELAMA KAMU MENGGUNAKAN WEBUI
import os, subprocess, time, urllib.request, re, signal, shutil, sys, threading

PORT = 8081

# ─── Fungsi helper: consume stdout cloudflared di thread ───
def consume_stdout(proc, on_line=None):
    """Baca stdout proc line-by-line, panggil callback, cetak ke console."""
    for line in iter(proc.stdout.readline, ''):
        print(line, end="", flush=True)
        if on_line:
            on_line(line)

# ─── Fungsi helper: kill proses di port tertentu ───
def kill_port(port):
    methods = [
        f"fuser -k {port}/tcp 2>/dev/null",
        f"lsof -ti:{port} | xargs kill -9 2>/dev/null",
        f"ss -tlnp | grep ':{port}' | tr -s ' ' | cut -d' ' -f7 | cut -d',' -f2 | xargs kill -9 2>/dev/null"
    ]
    for cmd in methods:
        ret = os.system(cmd)
        if os.WIFEXITED(ret) and os.WEXITSTATUS(ret) == 0:
            return True
    return False

# ─── 1. Cari binary open-webui ───
webui_bin = shutil.which("open-webui") or shutil.which("open_webui")
if not webui_bin:
    print("❌ open-webui tidak ditemukan. Jalankan Sel 1 dulu.")
    raise SystemExit(1)

# ─── 2. Kill port jika ada sisa ───
print(f"[1/4] Membersihkan port {PORT}...")
kill_port(PORT)
time.sleep(1)

# ─── 3. Start Open WebUI ───
print(f"[2/4] Memulai Open WebUI di port {PORT}...")
env = {**os.environ, "OLLAMA_BASE_URL": "http://127.0.0.1:11434"}
proc_webui = subprocess.Popen(
    [webui_bin, "serve", "--port", str(PORT)],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# ─── 4. Tunggu WebUI siap ───
print("[3/4] Menunggu WebUI siap (loading awal 1-3 menit)...")
webui_ready = False
for i in range(60):
    if proc_webui.poll() is not None:
        print(f"\n❌ WebUI gagal start (exit code {proc_webui.returncode})")
        raise SystemExit(1)
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{PORT}", timeout=5)
        r.close()
        print("\n✅ WebUI aktif di background!")
        webui_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

if not webui_ready:
    print("\n⚠️  Waktu tunggu habis — coba buka tunnel tetap dijalankan.")

# ─── 5. Cloudflare Tunnel (WebUI) ───
print("[4/4] Membuat link publik via Cloudflare Tunnel...\n")

webui_url = None

def on_webui_line(line):
    global webui_url
    m = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if m and webui_url is None:
        webui_url = m.group()

proc_tunnel_webui = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

t_webui = threading.Thread(
    target=consume_stdout,
    args=(proc_tunnel_webui, on_webui_line),
    daemon=True
)
t_webui.start()

# Tunggu URL tunnel (max 30 detik)
for _ in range(30):
    if webui_url:
        break
    time.sleep(1)
else:
    print("⚠️  Gagal mendapatkan URL tunnel untuk WebUI.")

if webui_url:
    print()
    print("═" * 60)
    print(" 🌐 BUKA LINK DI BAWAH INI PADA BROWSER ANDA:")
    print(f" 👉 {webui_url}")
    print("═" * 60)
    print("Cloudflare Tunnel langsung bekerja, tidak perlu klik 'Continue'.")

# ─── Keep-Alive: monitor kedua proses ───
print("\n📡 Memonitor proses WebUI & Tunnel (Ctrl+C untuk stop)...")
try:
    while True:
        time.sleep(30)
        if proc_webui.poll() is not None:
            print("⚠️  Proses WebUI telah berhenti! Jalankan ulang Sel 3.")
            break
        if proc_tunnel_webui.poll() is not None:
            print("⚠️  Tunnel Cloudflare terputus! Jalankan ulang Sel 3.")
            break
        print("✓ WebUI & Tunnel aktif —", time.strftime("%H:%M:%S"))
except KeyboardInterrupt:
    print("\nProses dihentikan secara manual.")

In [ ]:
# ── SEL 4: TUNNEL OLLAMA API + KONFIGURASI OPENCODE ───────
# Jalankan sel ini SETELAH Sel 3 (WebUI) sudah berjalan.
# Sel ini membuat tunnel untuk API Ollama (port 11434) agar bisa dipakai opencode CLI.
import subprocess, re, os, shutil, time, threading

OLLAMA_PORT = 11434

# ─── Fungsi helper ───
def consume_stdout(proc, on_line=None):
    for line in iter(proc.stdout.readline, ''):
        print(line, end="", flush=True)
        if on_line:
            on_line(line)

# ─── Verifikasi Ollama berjalan ───
import urllib.request
try:
    r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3)
    r.close()
    print("✅ Ollama API aktif di port 11434")
except Exception:
    print("⚠️  Ollama tidak merespon — pastikan Sel 2 sudah jalan.")
    # Tetap lanjut, mungkin akan nyambung nanti

# ─── 1. Tunnel untuk Ollama API ───
print("\n[1/2] Membuat tunnel untuk Ollama API...")

ollama_url = None

def on_ollama_line(line):
    global ollama_url
    m = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if m and ollama_url is None:
        ollama_url = m.group()

proc_tunnel_ollama = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{OLLAMA_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

t_ollama = threading.Thread(
    target=consume_stdout,
    args=(proc_tunnel_ollama, on_ollama_line),
    daemon=True
)
t_ollama.start()

# Tunggu URL tunnel (max 30 detik)
for _ in range(30):
    if ollama_url:
        break
    time.sleep(1)

if not ollama_url:
    print("\n❌ Gagal mendapatkan URL tunnel untuk Ollama API.")
    print("   Periksa: apakah cloudflared terinstall? Koneksi internet?")
else:
    print("═" * 60)
    print("🔗 OLLAMA API PUBLIC URL (untuk opencode):")
    print(f"   {ollama_url}")
    print("═" * 60)
    print()

    # ─── 2. Tampilkan konfigurasi opencode ───
    print("[2/2] Konfigurasi opencode.json")
    print("═" * 60)
    print("Buat file opencode.json di folder proyek Anda:\n")

    # Versi dengan /v1 (WAJIB untuk OpenAI-compatible API)
    print("{")
    print(f'  "endpoint": "{ollama_url}/v1",')
    print('  "model": "' + MODEL + '",')
    print('  "api_key": "",')
    print('  "provider": "openai"')
    print("}")
    print()
    print("📝 Catatan:")
    print("   - Tunnel Ollama API WAJIB berjalan agar opencode bisa terhubung.")
    print("   - Jika Colab disconnect, jalankan ulang Sel 2 → Sel 3 → Sel 4.")
    print("   - Ganti model sesuai yang dipilih di Sel 2.")
    print("   - ⚠️  URL ini bersifat PUBLIC — jangan bagikan ke orang lain!")

# ─── Keep-Alive ───
print("\n📡 Tunnel Ollama API aktif (Ctrl+C untuk stop)...")
try:
    while True:
        time.sleep(60)
        if proc_tunnel_ollama.poll() is not None:
            print("⚠️  Tunnel Ollama terputus! Jalankan ulang Sel 4.")
            break
        print("✓ Tunnel API —", time.strftime("%H:%M:%S"))
except KeyboardInterrupt:
    print("\nTunnel dihentikan.")